In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
import optuna
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import Adam
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import kagglehub
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
torch.cuda.empty_cache()

# Set up the device to be used for training (CUDA if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Download latest version
path = kagglehub.dataset_download("subhaditya/fer2013plus")

print("Path to dataset files:", path)

Using device: cuda
Path to dataset files: C:\Users\margi\.cache\kagglehub\datasets\subhaditya\fer2013plus\versions\1


In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="runs/fer2013plus")

# Download latest version
path = kagglehub.dataset_download("subhaditya/fer2013plus")

print("Path to dataset files:", path)
# Set up data transformations for training and testing
data_transforms = {
    "train": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomCrop(48, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]),
    "test": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
}

# Paths to dataset
train_dir = path + "/fer2013plus/fer2013/train"
test_dir = path + "/fer2013plus/fer2013/test"

# Load datasets using ImageFolder
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms["test"])

print("Train and Test Datasets Loaded!")

In [4]:



# Function to calculate class weights
def calculate_class_weights(train_dataset):
    class_counts = np.bincount(train_dataset.targets)
    class_weights = 1. / np.sqrt(class_counts)
    return torch.tensor(class_weights, dtype=torch.float32)


In [14]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import torch


# Function to create dataloaders with weighted sampling
def create_dataloader(batch_size):
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Calculate class weights to handle imbalanced dataset
    class_weights = calculate_class_weights(train_dataset)

    # Move class weights to the same device as the model
    class_weights = class_weights.to(device)

    # Create sample weights for training
    sample_weights = [class_weights[label] for label in train_dataset.targets]

    # Create WeightedRandomSampler for training
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    # Recreate the train loader with the sampler
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)

    return train_loader, test_loader, class_weights


In [15]:
import torch
import torch.optim as optim
import torch.nn.functional as F


# Function to train the model
def train_model(model, train_loader, optimizer, criterion, device, epochs=10, enable_logging=True):
    model.train()

    train_accuracy = []
    train_f1 = []
    train_losses = []

    for epoch in range(epochs):
        correct = 0
        total = 0
        all_preds = []
        all_labels = []
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Update running loss
            running_loss += loss.item()

            # Get predictions
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Store predictions and true labels for F1 calculation
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        # Calculate metrics for the epoch
        epoch_accuracy = 100 * correct / total
        epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
        epoch_loss = running_loss / len(train_loader)

        train_accuracy.append(epoch_accuracy)
        train_f1.append(epoch_f1)
        train_losses.append(epoch_loss)


        if enable_logging:
            # Log to TensorBoard
            writer.add_scalar('Accuracy/train', epoch_accuracy, epoch)
            writer.add_scalar('F1/train', epoch_f1, epoch)
            writer.add_scalar('Loss/train', epoch_loss, epoch)

        # Print epoch metrics
        print(f"Epoch {epoch + 1}/{epochs}, Train Accuracy: {epoch_accuracy:.2f}%, Train F1 Score: {epoch_f1:.4f}, Loss: {epoch_loss:.4f}")

    return train_accuracy, train_f1



In [16]:
def evaluate_model(model, test_loader, device, class_names):
    model.eval()

    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    # Initialize a dictionary to store class-wise accuracy using class names
    per_class_accuracy = {class_name: {"correct": 0, "total": 0} for class_name in class_names}

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            # Calculate overall accuracy
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Store predictions and true labels for F1 calculation
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Calculate per-class accuracy
            for label, prediction in zip(labels, predicted):
                class_name = class_names[label.item()]
                per_class_accuracy[class_name]["total"] += 1
                if label == prediction:
                    per_class_accuracy[class_name]["correct"] += 1

    # Calculate accuracy and F1 score for the entire dataset
    accuracy = 100 * correct / total
    f1 = f1_score(all_labels, all_preds, average='weighted')

    # Calculate per-class accuracy using class names
    class_accuracies = {
        class_name: (per_class_accuracy[class_name]["correct"] / per_class_accuracy[class_name]["total"]) * 100
        if per_class_accuracy[class_name]["total"] > 0 else 0
        for class_name in class_names}

    # Log test metrics to TensorBoard
    writer.add_scalar('Accuracy/test', accuracy, 0)
    writer.add_scalar('F1/test', f1, 0)

    # Log per-class accuracies
    for class_name, accuracy in class_accuracies.items():
        writer.add_scalar(f'Accuracy/test/class_{class_name}', accuracy, 0)

    print(f"Test Accuracy: {accuracy:.2f}%, Test F1 Score: {f1:.4f}")
    for class_name, accuracy in class_accuracies.items():
        print(f"Class {class_name} Accuracy: {accuracy:.2f}%")

    return class_accuracies  # Return per-class accuracies as a dictionary


In [17]:
## Define the CNN model
class CNN(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)

        # Fully connected layers
        self.fc1 = nn.Linear(in_features=128 * 6 * 6, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=128)
        self.fc3 = nn.Linear(in_features=128, out_features=8)

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)  # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [94]:
import optuna
from optuna.samplers import TPESampler


# Define the objective function for hyperparameter optimization
def objective(trial):
    # Define the hyperparameters to be tuned
    batch_size = trial.suggest_int('batch_size', 16, 128, step=16)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.5)
    epochs = trial.suggest_int('epochs', 3, 10)

    # Create DataLoader
    train_loader, test_loader, class_weights_tensor = create_dataloader(batch_size)

    # Create and move model to device
    model = CNN(dropout_rate).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    # Train the model
    train_model(model, train_loader, optimizer, criterion, device, epochs=epochs, enable_logging=False)

    # Evaluate the model and get per-class accuracy
    per_class_accuracy = evaluate_model(model, test_loader, device, train_dataset.classes)

    # Return the average accuracy across classes as the objective to be maximized
    return np.mean(list(per_class_accuracy.values()))  # Now it will work as per_class_accuracy is a dictionary


# Define a study to optimize the objective function
study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=10)  # You can increase n_trials for more exploration

# Print the best hyperparameters found
print("Best hyperparameters: ", study.best_params)




[I 2024-12-06 18:12:47,333] A new study created in memory with name: no-name-9d8551b9-4b7e-4f0b-a77a-23b487fb5db6


Epoch 1/5, Train Accuracy: 21.77%, Train F1 Score: 0.2189
Epoch 2/5, Train Accuracy: 28.00%, Train F1 Score: 0.2903
Epoch 3/5, Train Accuracy: 34.28%, Train F1 Score: 0.3542
Epoch 4/5, Train Accuracy: 38.42%, Train F1 Score: 0.3955
Epoch 5/5, Train Accuracy: 41.60%, Train F1 Score: 0.4273


[I 2024-12-06 18:14:44,400] Trial 0 finished with value: 43.86681709127528 and parameters: {'batch_size': 112, 'learning_rate': 0.0003749566595825611, 'dropout_rate': 0.27454269200567366, 'epochs': 5}. Best is trial 0 with value: 43.86681709127528.


Test Accuracy: 72.11%, Test F1 Score: 0.4932
Class 0 Accuracy: 36.65%
Class 1 Accuracy: 29.41%
Class 2 Accuracy: 38.60%
Class 3 Accuracy: 46.11%
Class 4 Accuracy: 71.10%
Class 5 Accuracy: 31.84%
Class 6 Accuracy: 25.12%
Class 7 Accuracy: 72.11%
Epoch 1/7, Train Accuracy: 16.72%, Train F1 Score: 0.1543
Epoch 2/7, Train Accuracy: 22.16%, Train F1 Score: 0.1981
Epoch 3/7, Train Accuracy: 23.32%, Train F1 Score: 0.2294
Epoch 4/7, Train Accuracy: 22.71%, Train F1 Score: 0.2355
Epoch 5/7, Train Accuracy: 24.66%, Train F1 Score: 0.2560
Epoch 6/7, Train Accuracy: 25.71%, Train F1 Score: 0.2683
Epoch 7/7, Train Accuracy: 26.02%, Train F1 Score: 0.2733


[I 2024-12-06 18:17:30,462] Trial 1 finished with value: 30.89653643257558 and parameters: {'batch_size': 96, 'learning_rate': 2.580358768501966e-05, 'dropout_rate': 0.4173599344759532, 'epochs': 7}. Best is trial 0 with value: 43.86681709127528.


...

Test Accuracy: 76.11%, Test F1 Score: 0.5763
Class 0 Accuracy: 51.40%
Class 1 Accuracy: 47.06%
Class 2 Accuracy: 50.88%
Class 3 Accuracy: 46.71%
Class 4 Accuracy: 69.24%
Class 5 Accuracy: 42.47%
Class 6 Accuracy: 41.47%
Class 7 Accuracy: 76.11%
Epoch 1/9, Train Accuracy: 22.23%, Train F1 Score: 0.1772
Epoch 2/9, Train Accuracy: 22.04%, Train F1 Score: 0.2152
Epoch 3/9, Train Accuracy: 23.29%, Train F1 Score: 0.2386
Epoch 4/9, Train Accuracy: 24.76%, Train F1 Score: 0.2543
Epoch 5/9, Train Accuracy: 24.94%, Train F1 Score: 0.2587
Epoch 6/9, Train Accuracy: 25.55%, Train F1 Score: 0.2647
Epoch 7/9, Train Accuracy: 27.53%, Train F1 Score: 0.2865
Epoch 8/9, Train Accuracy: 28.43%, Train F1 Score: 0.2957
Epoch 9/9, Train Accuracy: 28.38%, Train F1 Score: 0.2937


[I 2024-12-06 18:36:08,759] Trial 8 finished with value: 33.30895433108495 and parameters: {'batch_size': 64, 'learning_rate': 2.8470526486408083e-05, 'dropout_rate': 0.38554779765954117, 'epochs': 9}. Best is trial 7 with value: 53.1668082119795.


Test Accuracy: 58.44%, Test F1 Score: 0.2901
Class 0 Accuracy: 23.45%
Class 1 Accuracy: 60.78%
Class 2 Accuracy: 29.82%
Class 3 Accuracy: 34.73%
Class 4 Accuracy: 25.56%
Class 5 Accuracy: 14.17%
Class 6 Accuracy: 19.51%
Class 7 Accuracy: 58.44%
Epoch 1/3, Train Accuracy: 12.85%, Train F1 Score: 0.1412
Epoch 2/3, Train Accuracy: 22.91%, Train F1 Score: 0.2173
Epoch 3/3, Train Accuracy: 23.49%, Train F1 Score: 0.2338


[I 2024-12-06 18:37:24,847] Trial 9 finished with value: 22.08197391935267 and parameters: {'batch_size': 64, 'learning_rate': 1.8040256375121267e-05, 'dropout_rate': 0.3865261056296974, 'epochs': 3}. Best is trial 7 with value: 53.1668082119795.


Test Accuracy: 56.11%, Test F1 Score: 0.2440
Class 0 Accuracy: 32.61%
Class 1 Accuracy: 23.53%
Class 2 Accuracy: 1.75%
Class 3 Accuracy: 23.35%
Class 4 Accuracy: 17.57%
Class 5 Accuracy: 20.68%
Class 6 Accuracy: 1.05%
Class 7 Accuracy: 56.11%
Best hyperparameters:  {'batch_size': 32, 'learning_rate': 0.00019969325624696616, 'dropout_rate': 0.22158981707531503, 'epochs': 7}


In [18]:
# Step 1: Get the best hyperparameters from the optimization

# best_params = study.best_params
best_params = {'batch_size': 32, 'learning_rate': 0.00019969325624696616, 'dropout_rate': 0.22158981707531503,
               'epochs': 7}
batch_size = best_params[('batch_s'
                          'ize')]
learning_rate = best_params['learning_rate']
dropout_rate = best_params['dropout_rate']
epochs = best_params['epochs']

# Step 2: Create DataLoader with the best batch size
train_loader, test_loader, class_weights_tensor = create_dataloader(batch_size)

# Step 3: Create and initialize the model with the best dropout rate
model = CNN(dropout_rate).to(device)

# Step 4: Set up the optimizer and criterion (loss function)
optimizer = Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Step 5: Train the model with the best hyperparameters
train_accuracy, train_f1 = train_model(model, train_loader, optimizer, criterion, device, epochs=epochs)

# Step 6: Evaluate the model's performance
test_accuracy = evaluate_model(model, test_loader, device, train_dataset.classes)

writer.close()

Epoch 1/7, Train Accuracy: 22.34%, Train F1 Score: 0.2205, Loss: 1.9748
Epoch 2/7, Train Accuracy: 29.50%, Train F1 Score: 0.3060, Loss: 1.7773
Epoch 3/7, Train Accuracy: 36.41%, Train F1 Score: 0.3742, Loss: 1.6488
Epoch 4/7, Train Accuracy: 41.04%, Train F1 Score: 0.4202, Loss: 1.5321
Epoch 5/7, Train Accuracy: 43.79%, Train F1 Score: 0.4490, Loss: 1.4591
Epoch 6/7, Train Accuracy: 47.36%, Train F1 Score: 0.4830, Loss: 1.3559
Epoch 7/7, Train Accuracy: 49.33%, Train F1 Score: 0.5024, Loss: 1.3038
Test Accuracy: 70.78%, Test F1 Score: 0.5246
Class anger Accuracy: 50.31%
Class contempt Accuracy: 64.71%
Class disgust Accuracy: 54.39%
Class fear Accuracy: 49.70%
Class happiness Accuracy: 65.74%
Class neutral Accuracy: 30.34%
Class sadness Accuracy: 37.03%
Class surprise Accuracy: 70.78%


In [97]:
torch.save(model.state_dict(), 'cnn_strat_hyper.pth')